In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
import torchvision.transforms as transforms
import pandas as pd
from src.dataset import DistributedWeightedSampler, ImageDataset
from torch.utils.data import DataLoader, DistributedSampler
from torch import tensor
import torch
from metadata import metadata
from random import shuffle

md = metadata()
imgs_dir = '/projects/ag-bozek/nmoreau/dlbcl/data/normalized/'

In [2]:
#generate a first csv with all info -> then asked Martim to review images to select only images with a good quality
dots_hans_info = md.id_matching
dots_hans_info = dots_hans_info\
    .reset_index()\
    .loc[dots_hans_info.hans_classifier.isin(['ABC','GCB']), ['index', 'unique_id', 'hans_classifier', 'future_relapse_1yes_0no', 'tissue_type_1primary_2relapse', 'survival_months']]

dots_hans_info['filename'] = [os.path.join(imgs_dir, f'{i:03}_normalized.npy') for i in dots_hans_info.index]
dots_hans_info = dots_hans_info.loc[[os.path.exists(f) for f in dots_hans_info.filename]]
dots_hans_info["unique_id"] = dots_hans_info["unique_id"].str.strip()
dots_hans_info['patient_index'] = (dots_hans_info["unique_id"].str.extract(r"DLBCL(\d+)_"))
dots_hans_info["index_id"] = dots_hans_info["index"]

new_order = [
    "index_id",
    "patient_index",
    "unique_id",
    "hans_classifier",
    "future_relapse_1yes_0no",
    "tissue_type_1primary_2relapse",
    "survival_months",
    "filename",
]

dots_hans_info = dots_hans_info[new_order]
dots_hans_info.to_csv("dots_hans_info.csv", index=False)

In [3]:
#we now read Martim's file to extract only sample with a good quality
clinical_metadata = pd.read_csv('~/stefano_project/dot_hans_info_QCpass.csv',dtype='object').drop_duplicates()
clinical_metadata['filename'] = [os.path.join(imgs_dir,f'{int(i):03}_normalized.npy') for i in clinical_metadata.index_id]
new_order = [
    "index_id",
    "patient_index",
    "unique_id",
    "hans_classifier",
    "future_relapse_1yes_0no",
    "tissue_type_1primary_2relapse",
    "survival_months",
    "QC_pass",
    "filename",
]

clinical_metadata = clinical_metadata[new_order]

clinical_metadata = clinical_metadata[clinical_metadata["QC_pass"] == "True"]

In [4]:
#separation of samples in 3 folds in the same proportion of each class + samples from same patient in the same fold
from sklearn.model_selection import StratifiedGroupKFold

clinical_metadata["strat_label"] = (
    clinical_metadata["hans_classifier"].astype(str)
    + "_"
    + clinical_metadata["tissue_type_1primary_2relapse"].astype(str)
)

X = clinical_metadata              # features (can be the full DF)
y = clinical_metadata["hans_classifier"]
groups = clinical_metadata["patient_index"]

sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

clinical_metadata["fold"] = -1  # initialize

for fold, (_, val_idx) in enumerate(sgkf.split(X, y, groups)):
    clinical_metadata.iloc[val_idx, clinical_metadata.columns.get_loc("fold")] = fold

In [5]:
#check proportion
clinical_metadata.groupby("fold")["hans_classifier"].value_counts()

fold  hans_classifier
0     ABC                54
      GCB                36
1     ABC                54
      GCB                32
2     ABC                54
      GCB                30
Name: count, dtype: int64

In [6]:
clinical_metadata.groupby("fold")["tissue_type_1primary_2relapse"].value_counts()

fold  tissue_type_1primary_2relapse
0     1.0                              63
      2.0                              24
1     1.0                              64
      2.0                              22
2     1.0                              76
      2.0                               6
Name: count, dtype: int64

In [7]:
new_order = [
    "index_id",
    "patient_index",
    "unique_id",
    "fold",
    "hans_classifier",
    "future_relapse_1yes_0no",
    "tissue_type_1primary_2relapse",
    "survival_months",
    "filename",
]

clinical_metadata = clinical_metadata[new_order]

In [8]:
clinical_metadata.to_csv("training_metadata.csv", index=False)

In [9]:
clinical_metadata["hans_binary"] = (
    clinical_metadata["hans_classifier"]
    .map({"ABC": 0, "GCB": 1})
)

/tmp/ipykernel_666979/4283674101.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clinical_metadata["hans_binary"] = (


In [10]:
#training with fold 0 and 1 + val with fold 2
train_df = clinical_metadata[clinical_metadata["fold"].isin([0, 2])]
val_df   = clinical_metadata[clinical_metadata["fold"] == 1]

In [11]:
#check if patient are not in both training and validation
assert set(train_df["patient_index"]).isdisjoint(
    val_df["patient_index"]
)

In [12]:
#check proportion
train_df["hans_binary"].value_counts(normalize=True)

hans_binary
0    0.62069
1    0.37931
Name: proportion, dtype: float64

In [13]:
val_df["hans_binary"].value_counts(normalize=True)

hans_binary
0    0.627907
1    0.372093
Name: proportion, dtype: float64

In [14]:
new_order = [
    "index_id",
    "hans_binary",
    "filename",
]

train_df = train_df[new_order]
val_df = val_df[new_order]

In [15]:
train_df.to_csv("train_hans_bis.csv", index=False)

In [16]:
val_df.to_csv("val_hans_bis.csv", index=False)